# 01 - Data Preprocessing Pipeline 
## Mục tiêu
Làm sạch và chuẩn hóa 2.6 triệu giao dịch từ file `kz.csv` trước khi đưa vào phân tích **RFM** và **Streaming**.


### Các bước thực hiện:
1. Đọc dữ liệu bằng Spark DataFrame  
2. Kiểm tra chất lượng dữ liệu (Missing, Duplicate, Timestamp lỗi)  
3. Xử lý Missing Values – **giữ lại user_id null bằng cách tạo synthetic ID**  
4. Xử lý Duplicate  
5. Chuyển đổi kiểu dữ liệu  
6. Tạo cột mới (`total_amount`, `date`, `year_month`)  
7. Chuẩn hóa chuỗi (`brand`, `category`)  
8. Spark ML Pipeline (StringIndexer → VectorAssembler → StandardScaler)  
9. EDA trước/sau xử lý  
10. Lưu kết quả dạng Parquet

## 1. Import thư viện & Khởi tạo SparkSession

In [2]:
import os
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType,
    DoubleType, TimestampType, IntegerType
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, VectorAssembler, StandardScaler
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np

# ── SparkSession ──────────────────────────────────────────────────────────────
spark = (SparkSession.builder
         .appName("BCCK_Preprocessing_v2")
         .config("spark.sql.shuffle.partitions", "200")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")

Spark version : 3.5.0
Spark UI      : http://39d308d585bc:4040


## 2. Đọc dữ liệu CSV

In [3]:
schema = StructType([
    StructField("event_time",    StringType(),  True),
    StructField("order_id",      StringType(),  True),
    StructField("product_id",    StringType(),  True),
    StructField("category_id",   StringType(),  True),
    StructField("category_code", StringType(),  True),
    StructField("brand",         StringType(),  True),
    StructField("price",         DoubleType(),  True),
    StructField("user_id",       StringType(),  True),
])

DATA_PATH   = "/home/jovyan/work/data/kz.csv"
OUTPUT_PATH = "/home/jovyan/work/data/cleaned_orders.parquet"

df_raw = (spark.read
          .option("header", True)
          .option("inferSchema", False)
          .schema(schema)
          .csv(DATA_PATH))

total_raw = df_raw.count()
print(f"Tổng số giao dịch (raw)  : {total_raw:,}")
print(f"Số cột                   : {len(df_raw.columns)}")
df_raw.printSchema()
df_raw.show(5, truncate=False)

Tổng số giao dịch (raw)  : 2,633,521
Số cột                   : 8
root
 |-- event_time: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: string (nullable = true)

+-----------------------+-------------------+-------------------+-------------------+---------------------------+-------+------+-------------------+
|event_time             |order_id           |product_id         |category_id        |category_code              |brand  |price |user_id            |
+-----------------------+-------------------+-------------------+-------------------+---------------------------+-------+------+-------------------+
|2020-04-24 11:50:39 UTC|2294359932054536986|1515966223509089906|2268105426648170900|electronics.tablet         |samsung|162.01|1515915625441993984|
|2020-0

## 3. Kiểm tra chất lượng dữ liệu

> Phân tích toàn diện: Missing values, Duplicate, Timestamp bất thường, phân phối giá.

In [4]:
# ── 3.1 Missing Values ────────────────────────────────────────────────────────
print("=" * 55)
print("3.1  Missing Values")
print("=" * 55)

null_counts = {col: df_raw.filter(F.col(col).isNull()).count()
               for col in df_raw.columns}
null_df = pd.DataFrame(list(null_counts.items()),
                       columns=["column", "null_count"])
null_df["null_pct"] = (null_df["null_count"] / total_raw * 100).round(2)
null_df = null_df.sort_values("null_pct", ascending=False)
print(null_df.to_string(index=False))

# ── 3.2 Duplicate check ───────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("3.2  Duplicate check")
print("=" * 55)
dup_count = total_raw - df_raw.dropDuplicates(["event_time", "order_id", "product_id", "user_id"]).count()
print(f"Số dòng duplicate (toàn bộ): {dup_count:,}  ({dup_count/total_raw*100:.2f}%)")

# ── 3.3 Timestamp bất thường ─────────────────────────────────────────────────
print("\n" + "=" * 55)
print("3.3  Timestamp bất thường (1970 epoch)")
print("=" * 55)
df_ts_check = (df_raw
    .withColumn("ts", F.to_timestamp(F.regexp_replace("event_time", " UTC$", ""),
                                     "yyyy-MM-dd HH:mm:ss"))
    .withColumn("yr", F.year("ts")))
ts_dist = df_ts_check.groupBy("yr").count().orderBy("yr").toPandas()
print(ts_dist.to_string(index=False))
bad_ts = df_ts_check.filter(F.col("yr") < 2020).count()
print(f"\nDòng có timestamp < 2020 (lỗi epoch): {bad_ts:,}  ({bad_ts/total_raw*100:.2f}%)")

# ── 3.4 Phân phối giá ────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("3.4  Phân phối giá (price)")
print("=" * 55)
df_raw.select("price").describe().show()
neg_price = df_raw.filter(F.col("price") <= 0).count()
print(f"Dòng có price <= 0: {neg_price:,}")

3.1  Missing Values
       column  null_count  null_pct
      user_id     2069352     78.58
category_code      612202     23.25
        brand      506005     19.21
  category_id      431954     16.40
        price      431954     16.40
   event_time           0      0.00
     order_id           0      0.00
   product_id           0      0.00

3.2  Duplicate check
Số dòng duplicate (toàn bộ): 675  (0.03%)

3.3  Timestamp bất thường (1970 epoch)
  yr   count
1970   19631
2020 2613890

Dòng có timestamp < 2020 (lỗi epoch): 19,631  (0.75%)

3.4  Phân phối giá (price)
+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|           2201567|
|   mean| 154.0931653272589|
| stddev|241.94210909249253|
|    min|               0.0|
|    max|           50925.9|
+-------+------------------+

Dòng có price <= 0: 121


In [5]:
# ── Biểu đồ TRƯỚC khi xử lý ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Missing Values - TRƯỚC khi xử lý", fontsize=14, fontweight="bold")

colors = ["#e74c3c" if v > 0 else "#2ecc71" for v in null_df["null_count"]]
ax = axes[0]
bars = ax.barh(null_df["column"], null_df["null_count"], color=colors)
ax.set_xlabel("Số lượng null")
ax.set_title("Số lượng giá trị null theo cột")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for bar, val in zip(bars, null_df["null_count"]):
    if val > 0:
        ax.text(bar.get_width() + total_raw * 0.001,
                bar.get_y() + bar.get_height() / 2,
                f"{val:,}", va="center", fontsize=9)

ax2 = axes[1]
bars2 = ax2.barh(null_df["column"], null_df["null_pct"], color=colors)
ax2.set_xlabel("Tỷ lệ null (%)")
ax2.set_title("Tỷ lệ null (%) theo cột")
for bar, val in zip(bars2, null_df["null_pct"]):
    if val > 0:
        ax2.text(bar.get_width() + 0.1,
                 bar.get_y() + bar.get_height() / 2,
                 f"{val:.2f}%", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("missing_before.png", dpi=120, bbox_inches="tight")
plt.show()
print("Đã lưu: missing_before.png")

Đã lưu: missing_before.png


## 4. Xử lý Missing Values 

### Chiến lược mới:
| Cột | Hành động | Lý do |
|-----|-----------|-------|
| `price` | **Drop** | Không có giá = không phải giao dịch |
| `user_id` | **Giữ lại** → gán `order_user_<order_id>` | Có `order_id` là đủ để track giao dịch |
| `brand` | Điền `"unknown"` | Không ảnh hưởng phân tích chính |
| `category_code` | Điền `"unknown"` | Tương tự |
| `category_id` | Điền `"0"` | Tương tự |

>  **Lưu ý**: Các `user_id` synthetic dạng `order_user_xxx` sẽ được **tách riêng** trong bước RFM — chỉ dùng `user_id` thật cho phân tích khách hàng.

In [6]:
# ── Bước 1: Drop dòng không có price (bắt buộc) ─────────────────────────────
df_step1 = df_raw.dropna(subset=["price"])
cnt_after_price_drop = df_step1.count()
print(f"Sau khi drop price null     : {cnt_after_price_drop:,}  "
      f"(mất {total_raw - cnt_after_price_drop:,} dòng)")

# ── Bước 2: Loại timestamp lỗi (epoch 1970) ──────────────────────────────────
df_step2 = (df_step1
    .withColumn("_ts_check",
                F.to_timestamp(F.regexp_replace("event_time", " UTC$", ""),
                               "yyyy-MM-dd HH:mm:ss"))
    .filter(F.year("_ts_check") >= 2020)
    .drop("_ts_check")
)
cnt_after_ts_fix = df_step2.count()
print(f"Sau khi lọc timestamp lỗi   : {cnt_after_ts_fix:,}  "
      f"(mất {cnt_after_price_drop - cnt_after_ts_fix:,} dòng timestamp 1970)")

# ── Bước 3: Gán user_id synthetic cho null ────────────────────────────────────
# user_id null nhưng có order_id → dùng order_id làm định danh
df_step3 = (df_step2
    .withColumn("user_id",
        F.when(
            F.col("user_id").isNull() & F.col("order_id").isNotNull(),
            F.concat(F.lit("order_user_"), F.col("order_id"))
        ).when(
            F.col("user_id").isNull() & F.col("order_id").isNull(),
            F.lit("anonymous")
        ).otherwise(F.col("user_id"))
    )
    # Thêm flag để phân biệt user thật vs synthetic
    .withColumn("is_synthetic_user",
        F.when(F.col("user_id").startswith("order_user_"), F.lit(True))
         .when(F.col("user_id") == "anonymous", F.lit(True))
         .otherwise(F.lit(False)))
    # Điền null cho các cột khác
    .fillna({
        "brand":         "unknown",
        "category_code": "unknown",
        "category_id":   "0"
    })
)

cnt_after_fill = df_step3.count()
n_real_users = df_step3.filter(~F.col("is_synthetic_user")).count()
n_synthetic  = df_step3.filter(F.col("is_synthetic_user")).count()

print(f"\nSau khi xử lý null          : {cnt_after_fill:,}")
print(f"  - Dòng có user_id thật    : {n_real_users:,}  ({n_real_users/cnt_after_fill*100:.1f}%)")
print(f"  - Dòng user synthetic     : {n_synthetic:,}  ({n_synthetic/cnt_after_fill*100:.1f}%)")
print(f"\n>>> Giữ lại được {cnt_after_fill/total_raw*100:.1f}% dữ liệu gốc (so với v1: {563456/total_raw*100:.1f}%)")

Sau khi drop price null     : 2,201,567  (mất 431,954 dòng)
Sau khi lọc timestamp lỗi   : 2,186,014  (mất 15,553 dòng timestamp 1970)

Sau khi xử lý null          : 2,186,014
  - Dòng có user_id thật    : 562,862  (25.7%)
  - Dòng user synthetic     : 1,623,152  (74.3%)

>>> Giữ lại được 83.0% dữ liệu gốc (so với v1: 21.4%)


## 5. Xử lý Duplicate

In [7]:
cnt_before_dedup = df_step3.count()

KEY_COLS = ["event_time", "order_id", "product_id", "user_id"]
df_dedup = df_step3.dropDuplicates(KEY_COLS)

cnt_after_dedup = df_dedup.count()
removed_dup = cnt_before_dedup - cnt_after_dedup
dup_pct = removed_dup / cnt_before_dedup * 100

print(f"Trước khi dedup : {cnt_before_dedup:,}")
print(f"Sau khi dedup   : {cnt_after_dedup:,}")
print(f"Đã xóa          : {removed_dup:,} dòng ({dup_pct:.2f}%)")

fig, ax = plt.subplots(figsize=(7, 4))
labels = ["Sau null-drop (v2)", "Sau khi dedup"]
values = [cnt_before_dedup, cnt_after_dedup]
bars = ax.bar(labels, values, color=["#3498db", "#2ecc71"], width=0.4)
ax.set_title("Số dòng trước và sau khi loại Duplicate", fontweight="bold")
ax.set_ylabel("Số giao dịch")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + cnt_before_dedup * 0.005,
            f"{val:,}", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig("dedup_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Đã lưu: dedup_comparison.png")

Trước khi dedup : 2,186,014
Sau khi dedup   : 2,185,340
Đã xóa          : 674 dòng (0.03%)
Đã lưu: dedup_comparison.png


## 6. Chuyển đổi kiểu dữ liệu & Tạo cột mới

In [8]:
df_typed = (df_dedup
    # event_time: "2020-04-24 11:50:39 UTC" → timestamp
    .withColumn("event_time",
                F.to_timestamp(
                    F.regexp_replace("event_time", " UTC$", ""),
                    "yyyy-MM-dd HH:mm:ss"))
    # price → double
    .withColumn("price", F.col("price").cast(DoubleType()))
    # string casts
    .withColumn("user_id",    F.col("user_id").cast(StringType()))
    .withColumn("order_id",   F.col("order_id").cast(StringType()))
    .withColumn("product_id", F.col("product_id").cast(StringType()))
)

# Tạo cột mới
df_typed = (df_typed
    .withColumn("quantity",     F.lit(1).cast(IntegerType()))
    .withColumn("total_amount", F.col("price") * F.col("quantity"))
    .withColumn("date",
                F.date_trunc("day", F.col("event_time")).cast("date"))
    .withColumn("year_month",
                F.date_format(F.col("event_time"), "yyyy-MM"))
)

print("Schema sau khi chuyển đổi:")
df_typed.printSchema()
df_typed.select("event_time", "price", "quantity",
                "total_amount", "date", "year_month",
                "is_synthetic_user").show(5)

Schema sau khi chuyển đổi:
root
 |-- event_time: timestamp (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category_id: string (nullable = false)
 |-- category_code: string (nullable = false)
 |-- brand: string (nullable = false)
 |-- price: double (nullable = true)
 |-- user_id: string (nullable = true)
 |-- is_synthetic_user: boolean (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- total_amount: double (nullable = true)
 |-- date: date (nullable = true)
 |-- year_month: string (nullable = true)

+-------------------+-----+--------+------------+----------+----------+-----------------+
|         event_time|price|quantity|total_amount|      date|year_month|is_synthetic_user|
+-------------------+-----+--------+------------+----------+----------+-----------------+
|2020-04-29 12:27:47| 18.5|       1|        18.5|2020-04-29|   2020-04|            false|
|2020-04-29 13:31:33|74.05|       1|       74.05|2020-04-29|   

## 7. Chuẩn hóa chuỗi (String Processing)

In [9]:
df_str = (df_typed
    .withColumn("brand",
                F.trim(F.lower(F.col("brand"))))
    .withColumn("category",
                F.trim(F.lower(F.col("category_code"))))
    .withColumn("category_level1",
                F.split(F.col("category"), r"\.").getItem(0))
    # Lọc giá âm/bằng 0
    .filter(F.col("price") > 0)
)

cnt_final_str = df_str.count()
print(f"Dòng sau chuẩn hóa: {cnt_final_str:,}")

print("\nTop 10 brand:")
df_str.groupBy("brand").count().orderBy(F.desc("count")).show(10)

print("\nTop 10 category_level1:")
df_str.groupBy("category_level1").count().orderBy(F.desc("count")).show(10)

Dòng sau chuẩn hóa: 2,185,221

Top 10 brand:
+-------+------+
|  brand| count|
+-------+------+
|samsung|356343|
|    ava|117447|
|unknown|111892|
|  tefal| 78136|
|  apple| 74026|
| huawei| 56648|
|     lg| 55321|
|philips| 51894|
|    neo| 42315|
|polaris| 40419|
+-------+------+
only showing top 10 rows


Top 10 category_level1:
+---------------+------+
|category_level1| count|
+---------------+------+
|        unknown|606989|
|     appliances|605424|
|    electronics|544389|
|      computers|221270|
|      furniture|120447|
|     stationery| 39274|
|    accessories| 13516|
|   construction| 12527|
|        apparel|  7618|
|           kids|  4974|
+---------------+------+
only showing top 10 rows



## 8. EDA - Biểu đồ sau khi xử lý

In [10]:
df_pd = df_str.sample(False, 0.05, seed=42).toPandas()

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle("EDA - Dữ liệu SAU khi xử lý (v2)", fontsize=15, fontweight="bold")

# 1. Phân phối giá
ax = axes[0, 0]
df_pd["price"].clip(upper=df_pd["price"].quantile(0.99)).hist(
    bins=60, ax=ax, color="#3498db", edgecolor="white")
ax.set_title("Phân phối giá (cắt 99th percentile)")
ax.set_xlabel("Giá (USD)")
ax.set_ylabel("Tần suất")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# 2. Top 10 brand
ax = axes[0, 1]
brand_cnt = df_pd["brand"].value_counts().head(10)
brand_cnt.plot(kind="barh", ax=ax, color="#e67e22")
ax.set_title("Top 10 Brand theo số giao dịch (mẫu 5%)")
ax.set_xlabel("Số giao dịch")
ax.invert_yaxis()

# 3. Real vs Synthetic user
ax = axes[0, 2]
user_type = df_pd["is_synthetic_user"].value_counts()
labels_ut = ["User thật", "User synthetic"]
colors_ut = ["#27ae60", "#e74c3c"]
ax.pie(user_type.values, labels=labels_ut, autopct="%1.1f%%",
       colors=colors_ut, startangle=90)
ax.set_title("Tỷ lệ User thật vs Synthetic (mẫu 5%)")

# 4. Doanh thu theo tháng
ax = axes[1, 0]
monthly = (df_pd.groupby("year_month")["total_amount"]
           .sum().reset_index().sort_values("year_month"))
ax.bar(monthly["year_month"], monthly["total_amount"], color="#9b59b6")
ax.set_title("Doanh thu theo tháng (mẫu 5%)")
ax.set_xlabel("Tháng")
ax.set_ylabel("Total Amount (USD)")
ax.tick_params(axis="x", rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# 5. Phân bổ Category
ax = axes[1, 1]
cat_cnt = df_pd["category_level1"].value_counts().head(10)
explode = [0.05 if v/cat_cnt.sum() < 0.05 else 0 for v in cat_cnt.values]
wedges, texts, autotexts = ax.pie(
    cat_cnt.values,
    labels=None,
    autopct=lambda p: f"{p:.1f}%" if p > 3 else "",
    pctdistance=0.75,
    explode=explode,
    startangle=140,
    colors=plt.cm.Set3.colors,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}
)
for autotext in autotexts:
    autotext.set_fontsize(9)
    autotext.set_fontweight("bold")
ax.legend(wedges,
          [f"{k} ({v/cat_cnt.sum()*100:.1f}%)" for k, v in zip(cat_cnt.index, cat_cnt.values)],
          title="Category", title_fontsize=8,
          loc="upper left", bbox_to_anchor=(-0.3, 1.0), fontsize=8)
ax.set_title("Phân bổ Category cấp 1 (mẫu 5%)")

# 6. Số giao dịch theo tháng
ax = axes[1, 2]
txn_monthly = (df_pd.groupby("year_month").size()
               .reset_index(name="count").sort_values("year_month"))
ax.bar(txn_monthly["year_month"], txn_monthly["count"], color="#1abc9c")
ax.set_title("Số giao dịch theo tháng (mẫu 5%)")
ax.set_xlabel("Tháng")
ax.set_ylabel("Số giao dịch")
ax.tick_params(axis="x", rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.savefig("eda_after_processing.png", dpi=120, bbox_inches="tight")
plt.show()
print("Đã lưu: eda_after_processing.png")

Đã lưu: eda_after_processing.png


## 9. Spark ML Pipeline

3 bước:
1. **StringIndexer** – mã hóa `brand` và `category_level1` → số  
2. **VectorAssembler** – gom các cột số thành vector feature  
3. **StandardScaler** – chuẩn hóa về mean=0, std=1

In [11]:
# ── 9.1  StringIndexer ────────────────────────────────────────────────────────
brand_indexer    = StringIndexer(
    inputCol="brand",          outputCol="brand_idx",    handleInvalid="keep")
category_indexer = StringIndexer(
    inputCol="category_level1", outputCol="category_idx", handleInvalid="keep")

# ── 9.2  VectorAssembler ──────────────────────────────────────────────────────
feature_cols = ["price", "total_amount", "quantity", "brand_idx", "category_idx"]
assembler = VectorAssembler(
    inputCols=feature_cols, outputCol="features_raw", handleInvalid="keep")

# ── 9.3  StandardScaler ───────────────────────────────────────────────────────
scaler = StandardScaler(
    inputCol="features_raw", outputCol="features_scaled",
    withMean=True, withStd=True)

# ── Pipeline ──────────────────────────────────────────────────────────────────
pipeline = Pipeline(stages=[brand_indexer, category_indexer, assembler, scaler])

print("Đang fit Pipeline (StringIndexer + VectorAssembler + StandardScaler)...")
pipeline_model = pipeline.fit(df_str)
df_processed   = pipeline_model.transform(df_str)

print("Pipeline hoàn thành!")
df_processed.select("user_id", "is_synthetic_user", "event_time",
                     "price", "brand_idx", "category_idx",
                     "features_scaled").show(5, truncate=True)
print(f"Tổng số dòng sau Pipeline: {df_processed.count():,}")

Đang fit Pipeline (StringIndexer + VectorAssembler + StandardScaler)...
Pipeline hoàn thành!
+--------------------+-----------------+-------------------+------+---------+------------+--------------------+
|             user_id|is_synthetic_user|         event_time| price|brand_idx|category_idx|     features_scaled|
+--------------------+-----------------+-------------------+------+---------+------------+--------------------+
|order_user_234870...|             true|2020-01-05 03:43:28|143.98|     29.0|         0.0|[-0.0422313645842...|
|order_user_234871...|             true|2020-01-05 03:56:46| 22.89|     83.0|         2.0|[-0.5425512033488...|
|order_user_234871...|             true|2020-01-05 03:59:52|150.44|      0.0|         1.0|[-0.0155399271540...|
|order_user_234870...|             true|2020-01-05 04:14:15|  2.31|     30.0|         5.0|[-0.6275836773911...|
|order_user_234870...|             true|2020-01-05 04:14:15| 46.27|    134.0|         9.0|[-0.4459496852192...|
+----------

## 10. Biểu đồ số dòng qua từng bước xử lý

Tổng hợp số lượng giao dịch còn lại sau mỗi bước làm sạch dữ liệu, từ raw đến kết quả cuối cùng.

In [12]:
# ── Biểu đồ: Số dòng qua từng bước xử lý ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

stages = ["Raw", "Sau null-drop", "Sau dedup", "Sau filter price>0"]
counts = [total_raw, cnt_after_price_drop, cnt_after_dedup, cnt_final_str]
colors = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71"]

bars = ax.bar(stages, counts, color=colors, width=0.5)

ax.set_title("Số dòng qua từng bước xử lý", fontsize=14, fontweight="bold")
ax.set_ylabel("Số giao dịch")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

for bar, val in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(counts) * 0.01,
            f"{val:,}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("pipeline_stage_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Đã lưu: pipeline_stage_comparison.png")

Đã lưu: pipeline_stage_comparison.png


## 11. Lưu dữ liệu đã xử lý (Parquet)

In [13]:
COLS_TO_SAVE = [
    "user_id", "is_synthetic_user",   # is_synthetic_user → flag cho RFM
    "order_id", "product_id",
    "event_time", "date", "year_month",
    "brand", "brand_idx",
    "category", "category_level1", "category_idx",
    "price", "quantity", "total_amount",
    "features_scaled"
]

df_final = df_processed.select(*COLS_TO_SAVE)

(df_final
 .write
 .mode("overwrite")
 .partitionBy("year_month")
 .parquet(OUTPUT_PATH))

print(f" Đã lưu dữ liệu tại: {OUTPUT_PATH}")
print(f"   Tổng số dòng       : {df_final.count():,}")
print(f"   Số cột             : {len(COLS_TO_SAVE)}")

# Đọc lại xác nhận
df_verify = spark.read.parquet(OUTPUT_PATH)
print(f"\n Xác nhận đọc lại: {df_verify.count():,} dòng")
df_verify.printSchema()

 Đã lưu dữ liệu tại: /home/jovyan/work/data/cleaned_orders.parquet
   Tổng số dòng       : 2,185,221
   Số cột             : 16

 Xác nhận đọc lại: 2,185,221 dòng
root
 |-- user_id: string (nullable = true)
 |-- is_synthetic_user: boolean (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- brand: string (nullable = true)
 |-- brand_idx: double (nullable = true)
 |-- category: string (nullable = true)
 |-- category_level1: string (nullable = true)
 |-- category_idx: double (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- features_scaled: vector (nullable = true)
 |-- year_month: string (nullable = true)



## 12. Tóm tắt & Kết thúc

In [14]:
final_count = df_final.count()
n_real = df_final.filter(~F.col("is_synthetic_user")).count()
n_synth = df_final.filter(F.col("is_synthetic_user")).count()

print("=" * 65)
print("        TỔNG KẾT QUÁ TRÌNH TIỀN XỬ LÝ")
print("=" * 65)
print(f"  Dữ liệu gốc (raw)              : {total_raw:>12,} dòng")
print(f"  Sau drop price null             : {cnt_after_price_drop:>12,} dòng")
print(f"  Sau lọc timestamp lỗi (1970)   : {cnt_after_ts_fix:>12,} dòng")
print(f"  Sau gán user_id synthetic       : {cnt_after_fill:>12,} dòng")
print(f"  Sau dedup                       : {cnt_after_dedup:>12,} dòng")
print(f"  Sau filter price > 0 + Pipeline : {final_count:>12,} dòng")
print("-" * 65)
print(f"  → User có user_id thật          : {n_real:>12,} dòng  ({n_real/final_count*100:.1f}%)")
print(f"  → User synthetic (order_user_*) : {n_synth:>12,} dòng  ({n_synth/final_count*100:.1f}%)")
print("-" * 65)
print(f"  Giữ lại được                    : {final_count/total_raw*100:.1f}% dữ liệu gốc")
print("=" * 65)
print(f"  Output path  : {OUTPUT_PATH}")
print(f"  Output format: Parquet (partitioned by year_month)")
print(f"  Số cột       : {len(COLS_TO_SAVE)}")
print("=" * 65)
print("""
    Lưu ý khi sử dụng:
  - Phân tích RFM theo khách hàng thật: filter is_synthetic_user = False
  - Phân tích doanh thu / sản phẩm / streaming: dùng toàn bộ
  - Biểu đồ trend theo tháng: dùng toàn bộ
""")

spark.stop()
print(" SparkSession đã dừng.")

        TỔNG KẾT QUÁ TRÌNH TIỀN XỬ LÝ
  Dữ liệu gốc (raw)              :    2,633,521 dòng
  Sau drop price null             :    2,201,567 dòng
  Sau lọc timestamp lỗi (1970)   :    2,186,014 dòng
  Sau gán user_id synthetic       :    2,186,014 dòng
  Sau dedup                       :    2,185,340 dòng
  Sau filter price > 0 + Pipeline :    2,185,221 dòng
-----------------------------------------------------------------
  → User có user_id thật          :      562,149 dòng  (25.7%)
  → User synthetic (order_user_*) :    1,623,072 dòng  (74.3%)
-----------------------------------------------------------------
  Giữ lại được                    : 83.0% dữ liệu gốc
  Output path  : /home/jovyan/work/data/cleaned_orders.parquet
  Output format: Parquet (partitioned by year_month)
  Số cột       : 16

    Lưu ý khi sử dụng:
  - Phân tích RFM theo khách hàng thật: filter is_synthetic_user = False
  - Phân tích doanh thu / sản phẩm / streaming: dùng toàn bộ
  - Biểu đồ trend theo tháng: dùng